In [1]:
import pandas as pd

In [3]:
from pathlib import Path

folder = Path("data")
files_to_delete = ["2020.csv", "2021.csv", "2022.csv"]

for file_pattern in files_to_delete:
    for file_path in folder.rglob(file_pattern):
        print(f"Would delete: {file_path}")
        # Uncomment the line below to actually delete:
        file_path.unlink()

Would delete: data/West Bengal/Siliguri/Ward-32 Bapupara, Siliguri - WBPCB/2020.csv
Would delete: data/West Bengal/Kolkata/Ballygunge, Kolkata - WBPCB/2020.csv
Would delete: data/West Bengal/Kolkata/Bidhannagar, Kolkata - WBPCB/2020.csv
Would delete: data/West Bengal/Kolkata/Fort William, Kolkata - WBPCB/2020.csv
Would delete: data/West Bengal/Kolkata/Jadavpur, Kolkata - WBPCB/2020.csv
Would delete: data/West Bengal/Kolkata/Rabindra Bharati University, Kolkata - WBPCB/2020.csv
Would delete: data/West Bengal/Kolkata/Rabindra Sarobar, Kolkata - WBPCB/2020.csv
Would delete: data/West Bengal/Kolkata/Victoria, Kolkata - WBPCB/2020.csv
Would delete: data/West Bengal/Howrah/Belur Math, Howrah - WBPCB/2020.csv
Would delete: data/West Bengal/Howrah/Ghusuri, Howrah - WBPCB/2020.csv
Would delete: data/West Bengal/Howrah/Padmapukur, Howrah - WBPCB/2020.csv
Would delete: data/West Bengal/Asansol/Asansol Court Area, Asansol - WBPCB/2020.csv
Would delete: data/Uttarakhand/Dehradun/Doon University, De

In [5]:
from pathlib import Path
import pandas as pd
import shutil

root = Path("data")
dry_run = False  # Set to False to actually delete

na_tokens = ["NA", "N/A", "na", "null", "None", ""]

def station_has_only_na(station_dir: Path) -> bool:
    csv_files = list(station_dir.glob("*.csv"))
    if not csv_files:
        return True  # empty station folder

    found_valid_number = False

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file, na_values=na_tokens, low_memory=False)
        except Exception as e:
            print(f"Skipping unreadable file: {csv_file} ({e})")
            continue

        # Ignore time columns when computing max
        data_cols = [c for c in df.columns if "time" not in c.lower()]
        if not data_cols:
            continue

        numeric = df[data_cols].apply(pd.to_numeric, errors="coerce")
        overall_max = numeric.max(skipna=True).max(skipna=True)

        if pd.notna(overall_max):
            found_valid_number = True
            break

    return not found_valid_number


stations_checked = 0
stations_to_delete = []

for state_dir in root.iterdir():
    if not state_dir.is_dir():
        continue
    for city_dir in state_dir.iterdir():
        if not city_dir.is_dir():
            continue
        for station_dir in city_dir.iterdir():
            if not station_dir.is_dir():
                continue

            stations_checked += 1
            if station_has_only_na(station_dir):
                stations_to_delete.append(station_dir)

print(f"Stations checked: {stations_checked}")
print(f"Stations with all-NA data: {len(stations_to_delete)}")

for station_dir in stations_to_delete:
    if dry_run:
        print(f"[DRY RUN] Would delete: {station_dir}")
    else:
        shutil.rmtree(station_dir)
        print(f"Deleted: {station_dir}")

Stations checked: 572
Stations with all-NA data: 3
Deleted: data/Kerala/Ernakulam/Kacheripady, Ernakulam - Kerala PCB
Deleted: data/Madhya Pradesh/Ujjain/Mahashweta Nagar, Ujjain - MPPCB
Deleted: data/Uttar Pradesh/Raebareli/Indira Nagar, Raebareli - NTPC Unchahar


In [ ]:
from pathlib import Path
import pandas as pd

root = Path("data")
na_tokens = ["NA", "N/A", "na", "null", "None", ""]

records = []

for state_dir in root.iterdir():
    if not state_dir.is_dir():
        continue

    for city_dir in state_dir.iterdir():
        if not city_dir.is_dir():
            continue

        for station_dir in city_dir.iterdir():
            if not station_dir.is_dir():
                continue

            csv_files = sorted(station_dir.glob("*.csv"))
            if not csv_files:
                records.append({
                    "state": state_dir.name,
                    "city": city_dir.name,
                    "station": station_dir.name,
                    "files": 0,
                    "total_rows": 0,
                    "rows_with_any_data": 0,
                    "non_na_cells": 0,
                    "data_columns": 0,
                    "availability_percent": 0.0
                })
                continue

            total_rows = 0
            rows_with_any_data = 0
            non_na_cells = 0
            data_columns_max = 0

            for csv_file in csv_files:
                try:
                    df = pd.read_csv(csv_file, na_values=na_tokens, low_memory=False)
                except Exception as e:
                    print(f"Skipping unreadable file: {csv_file} ({e})")
                    continue

                # Ignore timestamp/time-like columns
                data_cols = [c for c in df.columns if "time" not in c.lower()]
                if not data_cols:
                    continue

                numeric = df[data_cols].apply(pd.to_numeric, errors="coerce")

                total_rows += len(numeric)
                rows_with_any_data += numeric.notna().any(axis=1).sum()
                non_na_cells += numeric.notna().sum().sum()
                data_columns_max = max(data_columns_max, numeric.shape[1])

            denom = total_rows * data_columns_max
            availability_percent = (non_na_cells / denom * 100.0) if denom > 0 else 0.0

            records.append({
                "state": state_dir.name,
                "city": city_dir.name,
                "station": station_dir.name,
                "files": len(csv_files),
                "total_rows": int(total_rows),
                "rows_with_any_data": int(rows_with_any_data),
                "non_na_cells": int(non_na_cells),
                "data_columns": int(data_columns_max),
                "availability_percent": round(availability_percent, 2)
            })

summary_df = pd.DataFrame(records).sort_values(["state", "city", "station"]).reset_index(drop=True)

# Full list in notebook
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
display(summary_df)

# Save report
out_file = Path("station_data_availability_summary.csv")
summary_df.to_csv(out_file, index=False)
print(f"Saved: {out_file} | Stations: {len(summary_df)}")

In [12]:
station_data = pd.read_csv("station_data_availability_summary.csv")

print(len(station_data))
print (len(station_data[station_data["availability_percent"] < 40.0]))

from pathlib import Path
import pandas as pd
import shutil

station_data = pd.read_csv("station_data_availability_summary.csv")

threshold = 40.0
dry_run = False  # keep True first, then set False to actually delete

to_remove = station_data[station_data["availability_percent"] < threshold]

removed = 0
missing = 0

for _, row in to_remove.iterrows():
    station_path = Path("data") / str(row["state"]) / str(row["city"]) / str(row["station"])

    if station_path.exists() and station_path.is_dir():
        if dry_run:
            print(f"[DRY RUN] Would delete: {station_path}")
        else:
            shutil.rmtree(station_path)
            print(f"Deleted: {station_path}")
            removed += 1
    else:
        print(f"Not found: {station_path}")
        missing += 1

print(f"Candidates below {threshold}%: {len(to_remove)}")
print(f"Deleted: {removed}")
print(f"Not found: {missing}")

569
104
Deleted: data/Andhra Pradesh/Machilipatnam/Srinivas Nagar Colony, Machilipatnam - APPCB
Deleted: data/Delhi/Delhi/Aya Nagar, Delhi - IMD
Deleted: data/Delhi/Delhi/Burari Crossing, Delhi - IMD
Deleted: data/Delhi/Delhi/CRRI Mathura Road, Delhi - IMD
Deleted: data/Delhi/Delhi/Cantonment Area, Delhi - DPCC
Deleted: data/Delhi/Delhi/Commonwealth Sports Complex, Delhi - DPCC
Deleted: data/Delhi/Delhi/IGI Airport (T3), Delhi - IMD
Deleted: data/Delhi/Delhi/IGNOU_Maidan Garhi, Delhi - DPCC
Deleted: data/Delhi/Delhi/IIT Delhi, Delhi - IITM
Deleted: data/Delhi/Delhi/ITO, Delhi - CPCB
Deleted: data/Delhi/Delhi/JNU, Delhi - DPCC
Deleted: data/Delhi/Delhi/Lodhi Road, Delhi - IITM
Deleted: data/Delhi/Delhi/NSUT Jaffarpur, Delhi - DPCC
Deleted: data/Delhi/Delhi/New Moti Bagh, Delhi - MHUA
Deleted: data/Delhi/Delhi/Pusa, Delhi - IMD
Deleted: data/Delhi/Delhi/Talkatora Garden, Delhi - DPCC
Deleted: data/Gujarat/Bhavnagar/Vidhyanagar, Bhavnagar - Nexteng Enviro
Deleted: data/Gujarat/Mehsana/Sad

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# =========================
# Config
# =========================
root = Path("data")
na_tokens = ["NA", "N/A", "na", "null", "None", ""]

# Keep rows having at least this fraction of non-NA values in data columns
# Example: 0.5 => remove rows where most columns are empty
min_non_na_ratio = 0.3

# If True: overwrite each source CSV with cleaned rows
# If False: only create combined cleaned file
write_cleaned_files = True

combined_out = Path("all_states_combined_cleaned.csv")
stats_out = Path("cleaning_stats_per_file.csv")

# =========================
# Processing
# =========================
all_parts = []
stats = []

for state_dir in sorted(root.iterdir()):
    if not state_dir.is_dir():
        continue

    for city_dir in sorted(state_dir.iterdir()):
        if not city_dir.is_dir():
            continue

        for station_dir in sorted(city_dir.iterdir()):
            if not station_dir.is_dir():
                continue

            for csv_file in sorted(station_dir.glob("*.csv")):
                try:
                    df = pd.read_csv(csv_file, na_values=na_tokens, low_memory=False)
                except Exception as e:
                    print(f"Skipping unreadable file: {csv_file} ({e})")
                    continue

                # Ignore time-like columns for emptiness check
                data_cols = [c for c in df.columns if "time" not in c.lower()]

                if data_cols:
                    numeric_view = df[data_cols].apply(pd.to_numeric, errors="coerce")
                    min_non_na = int(np.ceil(len(data_cols) * min_non_na_ratio))
                    keep_mask = numeric_view.notna().sum(axis=1) >= min_non_na
                    cleaned = df.loc[keep_mask].copy()
                else:
                    cleaned = df.copy()

                rows_before = len(df)
                rows_after = len(cleaned)
                rows_removed = rows_before - rows_after

                if write_cleaned_files:
                    cleaned.to_csv(csv_file, index=False)

                # Add provenance columns for final merge
                cleaned["state"] = state_dir.name
                cleaned["city"] = city_dir.name
                cleaned["station"] = station_dir.name
                cleaned["source_file"] = csv_file.name
                all_parts.append(cleaned)

                stats.append({
                    "state": state_dir.name,
                    "city": city_dir.name,
                    "station": station_dir.name,
                    "file": str(csv_file),
                    "rows_before": rows_before,
                    "rows_after": rows_after,
                    "rows_removed": rows_removed
                })

# =========================
# Save outputs
# =========================
final_df = pd.concat(all_parts, ignore_index=True, sort=False) if all_parts else pd.DataFrame()
final_df.to_csv(combined_out, index=False)

stats_df = pd.DataFrame(stats)
stats_df.to_csv(stats_out, index=False)

print(f"Processed files: {len(stats_df)}")
print(f"Combined rows: {len(final_df)}")
print(f"Combined file: {combined_out}")
print(f"Stats file: {stats_out}")
if len(stats_df) > 0:
    print(f"Total rows removed: {int(stats_df['rows_removed'].sum())}")
    display(stats_df.sort_values('rows_removed', ascending=False).head(20))

Processed files: 1369
Combined rows: 10313723
Combined file: all_states_combined_cleaned.csv
Stats file: cleaning_stats_per_file.csv
Total rows removed: 1677901


,state,city,station,file,rows_before,rows_after,rows_removed
218,Delhi,Delhi,"Lodhi Road, Delhi - IMD","data/Delhi/Delhi/Lodhi Road, Delhi - IMD/2024.csv",8784,0,8784
242,Delhi,Delhi,"North Campus, DU, Delhi - IMD","data/Delhi/Delhi/North Campus, DU, Delhi - IMD...",8784,0,8784
241,Delhi,Delhi,"North Campus, DU, Delhi - IMD","data/Delhi/Delhi/North Campus, DU, Delhi - IMD...",8760,0,8760
219,Delhi,Delhi,"Lodhi Road, Delhi - IMD","data/Delhi/Delhi/Lodhi Road, Delhi - IMD/2025.csv",8760,0,8760
243,Delhi,Delhi,"North Campus, DU, Delhi - IMD","data/Delhi/Delhi/North Campus, DU, Delhi - IMD...",8760,0,8760
787,Maharashtra,Pune,"Revenue Colony-Shivajinagar, Pune - IITM",data/Maharashtra/Pune/Revenue Colony-Shivajina...,8760,0,8760
217,Delhi,Delhi,"Lodhi Road, Delhi - IMD","data/Delhi/Delhi/Lodhi Road, Delhi - IMD/2023.csv",8760,0,8760
803,Maharashtra,Solapur,"Solapur, Solapur - MPCB","data/Maharashtra/Solapur/Solapur, Solapur - MP...",8784,25,8759
549,Madhya Pradesh,Indore,"Residency Area, Indore - IMC","data/Madhya Pradesh/Indore/Residency Area, Ind...",8760,79,8681
543,Madhya Pradesh,Indore,"Maguda Nagar, Indore - IMC","data/Madhya Pradesh/Indore/Maguda Nagar, Indor...",8760,129,8631


In [15]:
from pathlib import Path
import pandas as pd
import shutil

# Inputs
summary_path = Path("station_data_availability_summary.csv")
data_root = Path("data")

# Keep top N stations in each (state, city) group if the city is "data-dense"
top_n = 3

# "Data-dense city" = city having more than top_n stations
# (change to >= top_n if you want to enforce top_n for all cities with 3 or more)
dense_condition = "more_than_top_n"

# Safety switch
dry_run = False  # set False to actually delete

# Read summary
df = pd.read_csv(summary_path)

# Clean and robust sort keys
for col in ["availability_percent", "non_na_cells", "rows_with_any_data", "total_rows"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Rank stations inside each (state, city)
sort_cols = ["state", "city", "availability_percent", "non_na_cells", "rows_with_any_data", "total_rows", "station"]
ascending = [True, True, False, False, False, False, True]
ranked = df.sort_values(sort_cols, ascending=ascending).copy()
ranked["rank_in_city"] = ranked.groupby(["state", "city"]).cumcount() + 1
ranked["stations_in_city"] = ranked.groupby(["state", "city"])["station"].transform("count")

if dense_condition == "more_than_top_n":
    to_drop = ranked[(ranked["stations_in_city"] > top_n) & (ranked["rank_in_city"] > top_n)].copy()
elif dense_condition == "at_least_top_n":
    to_drop = ranked[(ranked["stations_in_city"] >= top_n) & (ranked["rank_in_city"] > top_n)].copy()
else:
    raise ValueError("dense_condition must be 'more_than_top_n' or 'at_least_top_n'")

# Build folder path and delete
deleted, missing = 0, 0
for _, row in to_drop.iterrows():
    station_path = data_root / str(row["state"]) / str(row["city"]) / str(row["station"])
    if station_path.exists() and station_path.is_dir():
        if dry_run:
            print(f"[DRY RUN] Would delete: {station_path}")
        else:
            shutil.rmtree(station_path)
            print(f"Deleted: {station_path}")
            deleted += 1
    else:
        print(f"Not found: {station_path}")
        missing += 1

# Optional report: kept stations
kept = ranked[~ranked.index.isin(to_drop.index)].copy()
kept_report = kept[["state", "city", "station", "rank_in_city", "availability_percent"]].sort_values(
    ["state", "city", "rank_in_city"]
)
kept_report.to_csv("kept_top3_stations_per_dense_city.csv", index=False)

print("\nSummary")
print(f"Total stations in summary: {len(df)}")
print(f"Stations marked for removal: {len(to_drop)}")
print(f"Deleted: {deleted}")
print(f"Not found: {missing}")
print("Kept report saved: kept_top3_stations_per_dense_city.csv")

Deleted: data/Assam/Guwahati/Pan Bazaar, Guwahati - PCBA
Deleted: data/Bihar/Patna/Samanpura, Patna - BSPCB
Deleted: data/Bihar/Patna/IGSC Planetarium Complex, Patna - BSPCB
Deleted: data/Chhattisgarh/Raipur/Siltara Phase-II, Raipur - CECB
Deleted: data/Delhi/Delhi/Dr. Karni Singh Shooting Range, Delhi - DPCC
Deleted: data/Delhi/Delhi/Okhla Phase-2, Delhi - DPCC
Deleted: data/Delhi/Delhi/Alipur, Delhi - DPCC
Deleted: data/Delhi/Delhi/Narela, Delhi - DPCC
Deleted: data/Delhi/Delhi/Patparganj, Delhi - DPCC
Deleted: data/Delhi/Delhi/Major Dhyan Chand National Stadium, Delhi - DPCC
Deleted: data/Delhi/Delhi/Ashok Vihar, Delhi - DPCC
Deleted: data/Delhi/Delhi/Sri Aurobindo Marg, Delhi - DPCC
Deleted: data/Delhi/Delhi/Mundka, Delhi - DPCC
Deleted: data/Delhi/Delhi/Pusa, Delhi - DPCC
Deleted: data/Delhi/Delhi/Vivek Vihar, Delhi - DPCC
Deleted: data/Delhi/Delhi/Najafgarh, Delhi - DPCC
Deleted: data/Delhi/Delhi/Jawaharlal Nehru Stadium, Delhi - DPCC
Deleted: data/Delhi/Delhi/Bawana, Delhi - DPC

In [21]:
from pathlib import Path
from difflib import SequenceMatcher
import pandas as pd
import re

# =========================
# Config
# =========================
data_root = Path("data")
meta_path = Path("stations_master.csv")
out_path = Path("all_states_combined_with_meta.csv")
match_report_path = Path("station_match_report.csv")
min_score = 0.30

# =========================
# Helpers
# =========================
def norm_text(x):
    x = "" if pd.isna(x) else str(x)
    x = x.replace("_", " ").replace("&", " and ")
    x = re.sub(r"\s+", " ", x.strip().lower())
    x = re.sub(r"[^a-z0-9 ,\-]", "", x)
    return x

def norm_state_city(x):
    # stations_master has state names like Andhra_Pradesh
    return norm_text(x).replace("_", " ")

def station_core(name):
    n = norm_text(name)
    # Keep the strongest identifying part of station name
    # e.g., "Gangineni Cheruvu, Chittoor - APPCB" -> "gangineni cheruvu"
    n = n.split(" - ")[0]
    n = n.split("-")[0]
    n = n.split(",")[0]
    return n.strip()

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# =========================
# Load station metadata
# =========================
meta = pd.read_csv(meta_path)

required_cols = {"station_id", "state", "city", "station_name", "latitude", "longitude"}
missing_cols = required_cols - set(meta.columns)
if missing_cols:
    raise ValueError(f"stations_master.csv missing columns: {missing_cols}")

meta_rows = []
for _, r in meta.iterrows():
    meta_rows.append({
        "station_id": r["station_id"],
        "state": r["state"],
        "city": r["city"],
        "station_name": r["station_name"],
        "latitude": r["latitude"],
        "longitude": r["longitude"],
        "state_norm": norm_state_city(r["state"]),
        "city_norm": norm_text(r["city"]),
        "station_norm": norm_text(r["station_name"]),
        "station_core": station_core(r["station_name"]),
    })

def best_station_match(folder_state, folder_city, folder_station, min_score=0.60):
    fs = norm_state_city(folder_state)
    fc = norm_text(folder_city)
    fstation_norm = norm_text(folder_station)
    fstation_core = station_core(folder_station)

    # 1) Prefer candidates from same state + city
    candidates = [m for m in meta_rows if m["state_norm"] == fs and m["city_norm"] == fc]

    # 2) Fallback to same state
    if not candidates:
        candidates = [m for m in meta_rows if m["state_norm"] == fs]

    # 3) Last fallback: all rows
    if not candidates:
        candidates = meta_rows

    best = None
    best_score = -1.0

    for m in candidates:
        s1 = similarity(fstation_norm, m["station_norm"])
        s2 = similarity(fstation_core, m["station_core"])
        score = 0.45 * s1 + 0.55 * s2
        if score > best_score:
            best_score = score
            best = m

    if best is not None and best_score >= min_score:
        return best, best_score
    return None, best_score

# =========================
# Merge all CSVs with metadata
# =========================
if out_path.exists():
    out_path.unlink()

match_cache = {}
match_rows = []
header_written = False

files_processed = 0
rows_written = 0
unreadable_files = 0

for state_dir in sorted(data_root.iterdir()):
    if not state_dir.is_dir():
        continue
    for city_dir in sorted(state_dir.iterdir()):
        if not city_dir.is_dir():
            continue
        for station_dir in sorted(city_dir.iterdir()):
            if not station_dir.is_dir():
                continue

            key = (state_dir.name, city_dir.name, station_dir.name)
            if key not in match_cache:
                best, score = best_station_match(
                    state_dir.name, city_dir.name, station_dir.name, min_score=min_score
                )
                if best is None:
                    match_cache[key] = {
                        "station_id": None,
                        "latitude": None,
                        "longitude": None,
                        "matched_station_name": None,
                        "match_score": float(score),
                    }
                else:
                    match_cache[key] = {
                        "station_id": best["station_id"],
                        "latitude": best["latitude"],
                        "longitude": best["longitude"],
                        "matched_station_name": best["station_name"],
                        "match_score": float(score),
                    }

            m = match_cache[key]

            for csv_file in sorted(station_dir.glob("*.csv")):
                try:
                    df = pd.read_csv(csv_file, low_memory=False)
                except Exception as e:
                    unreadable_files += 1
                    print(f"Skipping unreadable file: {csv_file} ({e})")
                    continue

                # Add folder context
                df["state"] = state_dir.name
                df["city"] = city_dir.name
                df["station"] = station_dir.name
                df["source_file"] = csv_file.name

                # Add matched metadata
                df["station_id"] = m["station_id"]
                df["latitude"] = m["latitude"]
                df["longitude"] = m["longitude"]
                df["matched_station_name"] = m["matched_station_name"]
                df["match_score"] = m["match_score"]

                df.to_csv(out_path, mode="a", header=not header_written, index=False)
                header_written = True

                files_processed += 1
                rows_written += len(df)

            match_rows.append({
                "state": state_dir.name,
                "city": city_dir.name,
                "station": station_dir.name,
                "station_id": m["station_id"],
                "latitude": m["latitude"],
                "longitude": m["longitude"],
                "matched_station_name": m["matched_station_name"],
                "match_score": m["match_score"],
            })

# Save station-level matching report
pd.DataFrame(match_rows).drop_duplicates(
    subset=["state", "city", "station"]
).sort_values(["state", "city", "station"]).to_csv(match_report_path, index=False)

print(f"Done. Files processed: {files_processed}")
print(f"Unreadable files skipped: {unreadable_files}")
print(f"Rows written: {rows_written}")
print(f"Combined file: {out_path}")
print(f"Match report: {match_report_path}")

Done. Files processed: 1024
Unreadable files skipped: 0
Rows written: 7786301
Combined file: all_states_combined_with_meta.csv
Match report: station_match_report.csv


In [ ]:
df = pd.read_csv("all_states_combined_with_meta.csv")
display(df.head())